In [1]:
# Lesson 5: Machine Learning
# Project: Placement Analytics & Prediction System

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

"""
train_test_split → splits data into train and test sets
LogisticRegression → first ML model we'll try
DecisionTreeClassifier → second ML model
RandomForestClassifier → third ML model
accuracy_score → measures how accurate our model is
classification_report → detailed performance report
.squeeze() → converts Y from DataFrame to Series"""

# Load prepared data
X = pd.read_csv('../data/processed/X_prepared.csv')
Y = pd.read_csv('../data/processed/Y_prepared.csv').squeeze()

print("✅ Libraries imported and data loaded!")
print(f"X shape: {X.shape}")
print(f"Y shape: {Y.shape}")

✅ Libraries imported and data loaded!
X shape: (215, 14)
Y shape: (215,)


In [ ]:
#Step 1: Train test split

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size = 0.2,        #20% data for testing, 80% for training
    random_state = 42       #reproducible split every time
)

print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_test.shape}")
print(f"\nTraining target: {Y_train.value_counts().to_dict()}") #Both sets have similar ratios of placed vs not placed 
print(f"Testing target: {Y_test.value_counts().to_dict()}") #this is called stratified splitting and train_test_split does this automatically. 

Training set size: (172, 14)
Testing set size: (43, 14)

Training target: {1: 117, 0: 55}
Testing target: {1: 31, 0: 12}


In [ ]:
#Step 2: Logistic Regression

lr_model = LogisticRegression(random_state= 42)         #creates the model
lr_model.fit(X_train, Y_train)               #trains the model on training data

lr_predictions = lr_model.predict(X_test)               #predicts placement for test students

lr_accuracy = accuracy_score(Y_test, lr_predictions)    #compares predictions with actual results
print(f"Logistic regression accuracy: {lr_accuracy * 100:.2f}%")

Logistic regression accuracy: 83.72%


In [ ]:
#Step 3: Detailed report

print("Classification Report - Logistic Regression")
print("\n")
print(classification_report(Y_test, lr_predictions, target_names=['Not Placed', 'Placed']))

"""
What classification_report shows:
Precision   → out of all students predicted as Placed, how many actually got placed?
Recall      → out of all students who actually got placed, how many did we correctly predict?
F1 Score    → balance between Precision and Recall
Support     → actual number of students in each class"""

Classification Report - Logistic Regression
              precision    recall  f1-score   support

  Not Placed       0.78      0.58      0.67        12
      Placed       0.85      0.94      0.89        31

    accuracy                           0.84        43
   macro avg       0.82      0.76      0.78        43
weighted avg       0.83      0.84      0.83        43



In [7]:
#Step 4: Decision Tree

dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, Y_train)

dt_predictions = dt_model.predict(X_test)

dt_accuracy = accuracy_score(Y_test, dt_predictions)
print(f"Decision Tree Accuracy: {dt_accuracy * 100:.2f}%")

print("Classificatiion report: Decision tree")
print("\n")
print(classification_report(Y_test, dt_predictions, target_names=['Not Placed', 'Placed']))

Decision Tree Accuracy: 83.72%
Classificatiion report: Decision tree


              precision    recall  f1-score   support

  Not Placed       0.69      0.75      0.72        12
      Placed       0.90      0.87      0.89        31

    accuracy                           0.84        43
   macro avg       0.80      0.81      0.80        43
weighted avg       0.84      0.84      0.84        43



In [ ]:
#Step 5: Random Forest 

rf_model = RandomForestClassifier(n_estimators = 100, random_state=42)
rf_model.fit(X_train, Y_train)

rf_predictions = rf_model.predict(X_test)

rf_accuracy = accuracy_score(Y_test, rf_predictions)
print(f"Random forest accuracy: {rf_accuracy * 100:.2f}%")

print("\nClassifiaction Report - Random Forest")
print("\n")
print(classification_report(Y_test, rf_predictions, target_names=['Not Placed', 'Placed']))

Random forest accuracy: 76.74%


Classifiaction Report - Random Forest


              precision    recall  f1-score   support

  Not Placed       0.67      0.33      0.44        12
      Placed       0.78      0.94      0.85        31

    accuracy                           0.77        43
   macro avg       0.73      0.63      0.65        43
weighted avg       0.75      0.77      0.74        43



In [10]:
#Step 6: Check for overfitting

models = {
    'Logistic Regression' : lr_model,
    'Decision Tree' : dt_model,
    'Random Forest' : rf_model
}

print("Overfitting Check")
for name, model in models.items():
    train_acc = accuracy_score(Y_train, model.predict(X_train))
    test_acc = accuracy_score(Y_test, model.predict(X_test))
    gap = train_acc - test_acc

    print(f"\n{name}: ")

    print(f"Train accuracy: {train_acc * 100:.2f}%")
    print(f"Test accuracy: {test_acc * 100:.2f}%")
    print(f"Gap : {gap * 100:.2f}%")

    if gap > 0.1:
        print("Overfitting Detected")
    else :
        print("Generalizing Well")



Overfitting Check

Logistic Regression: 
Train accuracy: 88.95%
Test accuracy: 83.72%
Gap : 5.23%
Generalizing Well

Decision Tree: 
Train accuracy: 100.00%
Test accuracy: 83.72%
Gap : 16.28%
Overfitting Detected

Random Forest: 
Train accuracy: 100.00%
Test accuracy: 76.74%
Gap : 23.26%
Overfitting Detected


In [12]:
#Step 7: Save final model

import pickle                                               #Python library that converts any object into a file
with open('../models/placement_model.pkl', 'wb') as f:     #opens file in write binary mode
    pickle.dump(lr_model, f)                                #saves the model into the file
#.. → go one folder up (from notebooks to placement-predictor)
#... → Python doesn't understand this as a path!
#.pkl → standard extension for saved Python objects

print("Model saved to model/placement_model.pkl")

Model saved to model/placement_model.pkl
